# Chakra ViT fix: 0. diagnostic (about 15 minutes)

Trains the **patched ViT-B for 5 epochs** twice at the same time: arm A in fp16 on GPU 0, arm B in bf16 on GPU 1. On 29 Sep three ViT-B runs trained in emulated bf16 on T4s and never learned (val 0.32-0.44 at epoch 1, then 0.00).

**Verdict** at the end: arm A best val mDice >= 0.65 = PASS -> run RUN_A, RUN_B, RUN_C. Below 0.5 = FAIL -> stop and send the SEND_BACK zips.

Settings: **GPU T4 x2**, **Internet ON**. Inputs: the uploaded `chakra_vit_fix` dataset, `chakra-leakbench-v1`, `chakra-leakbench-e4-v1`. Run all cells; nothing to edit.

In [ ]:
# ---- 1. find this bundle and the data (works with /kaggle/input/<slug> and /kaggle/input/datasets/<owner>/<slug>)
import os, sys, glob, json, time, shutil, subprocess, zipfile
EXTRA_ROOTS = []      # lab PC only: e.g. [r"M:/chakramodelpro"]
SMOKE = os.environ.get("CHAKRA_SMOKE") == "1"   # developer switch: tiny random models (leave off)

hits = sorted(glob.glob("/kaggle/input/**/CHAKRA_BUNDLE_ID", recursive=True)) + \
       sorted(glob.glob(os.path.join(os.getcwd(), "**", "CHAKRA_BUNDLE_ID"), recursive=True))
assert hits, "bundle not found: add the uploaded chakra_vit_fix dataset as an Input"
BUNDLE = os.path.dirname(hits[0]); sys.path.insert(0, BUNDLE)
print("bundle:", BUNDLE, "|", open(hits[0]).read().strip())
import chakra_env
if EXTRA_ROOTS:
    roots, found = chakra_env.find_roots(EXTRA_ROOTS)
else:
    roots, found = chakra_env.find_roots()
for n in chakra_env.NEEDED:
    print(f"  {n}: {'found in ' + found[n] if n in found else 'MISSING'}")
assert roots and all(n in found for n in chakra_env.NEEDED), \
    "data not found: add BOTH chakra-leakbench-v1 and chakra-leakbench-e4-v1 as Inputs"
for m in ("v2.csv", "r6_mixneg.csv"):
    checked, missing = chakra_env.verify(os.path.join(BUNDLE, "manifests", m), roots)
    print(f"manifest {m}: {checked} rows sampled, {len(missing)} unresolved", missing[:2])
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE")
if N_GPU == 0 and not SMOKE:
    raise SystemExit("No GPU: Settings -> Accelerator -> GPU T4 x2")
W = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
LOGS = os.path.join(W, "logs"); os.makedirs(LOGS, exist_ok=True)

In [ ]:
# ---- 3. launch: one job per GPU when there are 2 jobs and 2 GPUs; otherwise one after the other
JOBS = {
    "v2_s42": ("v2.csv", 42), "v2_s43": ("v2.csv", 43), "v2_s44": ("v2.csv", 44),
    "r6_s42": ("r6_mixneg.csv", 42), "r6_s43": ("r6_mixneg.csv", 43), "r6_s44": ("r6_mixneg.csv", 44),
}
RECIPE = ["--epochs", "50", "--size", "352", "--bs", "-1", "--target-batch", "16",
          "--lr", "1e-4", "--llrd", "0.8", "--drop-path", "0.2", "--warmup-frac", "0.06", "--clip", "1.0",
          "--workers", "4", "--fast", "--amp-dtype", "auto", "--abort-epoch", "5", "--abort-below", "0.5"]


def job_cmd(man, seed, gpus, extra=()):
    cmd = [sys.executable, os.path.join(BUNDLE, "train.py"), "--data-root", *roots,
           "--manifest", os.path.join(BUNDLE, "manifests", man), "--model", MODEL, "--seed", str(seed),
           "--gpus", str(gpus), "--out", os.path.join(W, "runs"), *RECIPE, *extra]
    if SMOKE:
        cmd += ["--no-pretrained", "--size", "64", "--epochs", "2", "--workers", "0", "--abort-epoch", "0"]
    return cmd


def launch(jobs):
    parallel = len(jobs) > 1 and N_GPU >= len(jobs)
    procs = []
    for i, (name, cmd) in enumerate(jobs):
        env = dict(os.environ)
        if parallel:
            env["CUDA_VISIBLE_DEVICES"] = str(i)
        log = open(os.path.join(LOGS, name + ".log"), "w")
        print(f"start {name} on {'GPU ' + str(i) if parallel else 'all GPUs'}:", " ".join(cmd[1:])[:200], flush=True)
        p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
        procs.append((name, p, log))
        if not parallel:
            watch([procs[-1]])
    if parallel:
        watch(procs)
    return {n: p.returncode for n, p, _ in procs}


def last_epoch(name):
    last = None
    for line in open(os.path.join(LOGS, name + ".log"), errors="ignore"):
        if line.startswith("{'epoch'"):
            last = line.strip()
    return last


def watch(procs, every=120):
    t0 = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(2 if SMOKE else every)
        for name, p, _ in procs:
            print(f"[{(time.time() - t0) / 60:6.1f} min] {name}: {'running' if p.poll() is None else 'exit ' + str(p.returncode)} | {last_epoch(name)}", flush=True)
    for name, p, log in procs:
        log.close()
        tail = open(os.path.join(LOGS, name + ".log"), errors="ignore").read()[-1500:]
        print(f"==== {name} exit {p.returncode}\n{tail}", flush=True)

In [ ]:
# ---- 3b. the diagnostic: patched ViT-B, 5 epochs, r6_mixneg; arm A fp16 on GPU 0, arm B bf16 on GPU 1
MODEL = "vit_dpt_b"
DIAG = ["--epochs", "5", "--size", "352", "--bs", "8", "--accum", "2", "--lr", "1e-4", "--llrd", "0.8", "--drop-path", "0.2",
        "--warmup-frac", "0.06", "--clip", "1.0", "--workers", "4", "--fast"]
jobs = []
for name, dt in (("A_fp16", "fp16"), ("B_bf16", "bf16")):
    cmd = [sys.executable, os.path.join(BUNDLE, "train.py"), "--data-root", *roots,
           "--manifest", os.path.join(BUNDLE, "manifests", "r6_mixneg.csv"), "--model", MODEL, "--seed", "42",
           "--gpus", "1", "--out", os.path.join(W, "diag", name), *DIAG, "--amp-dtype", dt]
    if SMOKE:
        cmd += ["--no-pretrained", "--size", "64", "--epochs", "2", "--workers", "0"]
    jobs.append((name, cmd))
codes = launch(jobs)
print(codes)

In [ ]:
# ---- 3c. verdict ----------------------------------------------------------------
import ast
def curve(name):
    return [ast.literal_eval(l.strip())["val_mDice"] for l in open(os.path.join(LOGS, name + ".log"), errors="ignore")
            if l.startswith("{'epoch'")]
best = {}
for name in ("A_fp16", "B_bf16"):
    v = curve(name); best[name] = max(v) if v else float("nan")
    print(f"{name}: val mDice by epoch {[round(x, 3) for x in v]} -> best {best[name]:.3f}")
a, b = best["A_fp16"], best["B_bf16"]
if a >= 0.65:
    print("\nPASS: the fixed ViT-DPT learns in fp16 on this GPU. Run RUN_A, RUN_B and RUN_C next.")
elif a < 0.5:
    print("\nFAIL: it still does not learn in fp16. Do NOT start RUN_A/B/C. Send back SEND_BACK_*.zip.")
else:
    print("\nINCONCLUSIVE (0.50-0.65): rerun this notebook once; if it repeats, send the logs back.")
if a >= 0.65:
    print("bf16 arm:", "bf16 WAS a cause of the collapse" if b < 0.5 else "bf16 alone does not explain the collapse")
# the diag runs live under diag/, so pack them for the report cell
os.makedirs(os.path.join(W, "runs"), exist_ok=True)
for name in ("A_fp16", "B_bf16"):
    for rd in glob.glob(os.path.join(W, "diag", name, "*")):
        dst = os.path.join(W, "runs", f"DIAG_{name}__" + os.path.basename(rd))
        if not os.path.exists(dst):
            shutil.copytree(rd, dst, ignore=shutil.ignore_patterns("*.pt"))

In [ ]:
# ---- 4. results + SEND_BACK zip (no weights) --------------------------------------
def report_and_pack():
    for rd in sorted(glob.glob(os.path.join(W, "runs", "*"))):
        tag = os.path.basename(rd)
        ab = os.path.join(rd, "results_ABORTED.json")
        if os.path.exists(ab):
            print(f"!! {tag} ABORTED by the sanity gate: {json.load(open(ab))['reason']}")
        rp = os.path.join(rd, "results.json")
        if os.path.exists(rp):
            r = json.load(open(rp))
            print(f"== {tag} | best val {r['best_val_mDice']:.4f} | amp {r.get('amp_dtype')} | batch {r['bs']}x{r['accum']}"
                  f"={r['effective_batch']} | {r['n_gpu']} GPU | {r['train_minutes']:.0f} min | input {r.get('input_norm')}")
            for k in sorted(r["tests"]):
                t = r["tests"][k]
                if t.get("skipped") or not t.get("n"):
                    print(f"   {k:18s} SKIPPED ({t.get('skipped', 'no rows')})")
                    continue
                print(f"   {k:18s} n={t['n']:5d} " + (f"false alarm @0.001 = {t['false_alarm_rate@0.001']:.4f}" if k.startswith("neg_")
                                                        else f"mDice {t['mDice']:.4f}  small {t['mDice_small']}"))
            if r.get("dropped_eval_rows"):
                print("   dropped eval rows:", {k: len(v) for k, v in r["dropped_eval_rows"].items()})
        zp = os.path.join(W, f"SEND_BACK_{tag}.zip")
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            for f in glob.glob(os.path.join(rd, "*")):
                if not f.endswith(".pt"):
                    z.write(f, os.path.join(tag, os.path.basename(f)))
            for f in glob.glob(os.path.join(LOGS, "*.log")):
                z.write(f, os.path.join(tag, "logs", os.path.basename(f)))
        print("   ->", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")


report_and_pack()

**Download** `SEND_BACK_*.zip` from the Output tab (or `/kaggle/working`) and send it back. The `best.pt` weights stay in the Output if they are needed later.